# 14 — Seed-variance check on Phase 10 augmentation

**Goal.** Confirm that the Phase 10 result (augmentation lifting overall macro-F1 from 0.550 to 0.603) is not a one-seed artifact, and provide a reference point for interpreting the no-AKIEC ablation in notebook 13.

**Setup.** Exact same recipe as `09_train_augmented.ipynb`:

- frozen ResNet50 backbone
- batch 32, LR 1e-3, 40 epochs, patience 7
- `SOURCES_PER_CLASS = {2:'cvae', 3:'cvae', 5:'cvae', 6:'cvae'}` (BCC, AKIEC, DF, VASC all augmented)
- class-weighted CE loss from real train counts

**One change:** `SEED = 123` instead of `SEED = 42`. Everything else identical.

**How to read the result.** If macro-F1 lands within ~2 points of the original 0.603, the Phase 10 finding is robust to seed and the no-AKIEC drop in notebook 13 (0.486) is a real degradation, not noise. If it lands more than 3 points from 0.603 in either direction, the project's per-experiment differences need to be interpreted as noisy and the report should explicitly say so.

**This is the last experiment.** After this, no more training runs — the remaining time is for the report.

## 1. Imports, paths, seed

In [ ]:
import sys
from pathlib import Path
import csv
import time
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, balanced_accuracy_score, recall_score,
)

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.dataset import ISICDataset, CLASS_NAMES, get_class_weights        # noqa: E402
from src.dataset_augmented import AugmentedISICDataset                      # noqa: E402
from src.models.classifier import build_resnet50_classifier                 # noqa: E402

SPLITS_CSV = PROJECT_ROOT / 'data' / 'processed' / 'splits.csv'
IMAGES_DIR = PROJECT_ROOT / 'data' / 'raw' / 'ISIC2018_Task3_Training_Input'
MANIFEST   = PROJECT_ROOT / 'data' / 'synthetic' / 'synthetic_manifest.csv'

RESULTS    = PROJECT_ROOT / 'results'
CKPT_DIR   = RESULTS / 'checkpoints'
LOGS_DIR   = RESULTS / 'logs'
PLOTS_DIR  = RESULTS / 'plots'
for d in (CKPT_DIR, LOGS_DIR, PLOTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

assert SPLITS_CSV.exists(), f'splits.csv not found at {SPLITS_CSV}'
assert MANIFEST.exists(),   f'synthetic_manifest.csv not found at {MANIFEST}'

# THE ONE CHANGE: seed is 123, not 42
SEED = 123
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print(f'Seed fixed to {SEED}  (Phase 10 used 42)')

## 2. Device check

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 3. Hyperparameters

Identical to Phase 10.

In [ ]:
IMG_SIZE      = 224
BATCH_SIZE    = 32
EPOCHS        = 40
LR            = 1e-3
PATIENCE      = 7
NUM_WORKERS   = 2
N_CLASSES     = len(CLASS_NAMES)

SOURCES_PER_CLASS = {
    2: 'cvae',    # BCC
    3: 'cvae',    # AKIEC
    5: 'cvae',    # DF
    6: 'cvae',    # VASC
}

print(f'image size      : {IMG_SIZE}')
print(f'batch size      : {BATCH_SIZE}')
print(f'epochs          : {EPOCHS}')
print(f'lr              : {LR}')
print(f'patience        : {PATIENCE}')
print(f'sources per cls : {SOURCES_PER_CLASS}')

## 4. Transforms

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

## 5. Datasets and loaders

In [ ]:
train_ds = AugmentedISICDataset(
    splits_csv=SPLITS_CSV, images_dir=IMAGES_DIR,
    synth_manifest_csv=MANIFEST, project_root=PROJECT_ROOT,
    split='train', transform=train_tf,
    sources_per_class=SOURCES_PER_CLASS,
)
val_ds  = ISICDataset(SPLITS_CSV, IMAGES_DIR, split='val',  transform=eval_tf)
test_ds = ISICDataset(SPLITS_CSV, IMAGES_DIR, split='test', transform=eval_tf)

print(f'train  : {len(train_ds):>6,}')
print(f'val    : {len(val_ds):>6,}')
print(f'test   : {len(test_ds):>6,}')

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

## 6. Class weights

In [ ]:
class_weights = get_class_weights(SPLITS_CSV, split='train').to(DEVICE)
for i, name in enumerate(CLASS_NAMES):
    print(f'  {name:<6} idx={i}  weight={class_weights[i].item():.4f}')

## 7. EarlyStopping

In [ ]:
class EarlyStopping:
    # Stops training when val loss stops improving for `patience` epochs.

    def __init__(self, patience: int = 5, delta: float = 0.0):
        self.patience = patience
        self.delta = delta
        self.best_loss = None
        self.no_improvement_count = 0
        self.stop_training = False

    def check_early_stop(self, val_loss: float):
        if self.best_loss is None or val_loss < self.best_loss - self.delta:
            self.best_loss = val_loss
            self.no_improvement_count = 0
        else:
            self.no_improvement_count += 1
            if self.no_improvement_count >= self.patience:
                self.stop_training = True

## 8. Train / evaluate helpers

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0; correct = 0; total = 0
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        correct += (logits.argmax(dim=1) == y).sum().item()
        total += x.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0; correct = 0; total = 0
    all_preds = []; all_labels = []
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        logits = model(x)
        loss = criterion(logits, y)
        total_loss += loss.item() * x.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += x.size(0)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(y.cpu().tolist())
    return (total_loss / total, correct / total,
            f1_score(all_labels, all_preds, average='macro', zero_division=0),
            balanced_accuracy_score(all_labels, all_preds),
            all_preds, all_labels)

## 9. Model and optimiser

In [ ]:
model = build_resnet50_classifier(num_classes=N_CLASSES, freeze_backbone=True).to(DEVICE)
trainable = [p for p in model.parameters() if p.requires_grad]
print(f'trainable params: {sum(p.numel() for p in trainable):,}')

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(trainable, lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3,
)

## 10. Training loop

In [ ]:
EXPERIMENT_NAME = 'augmented_seed123'
log_path  = LOGS_DIR / f'{EXPERIMENT_NAME}_log.csv'
ckpt_path = CKPT_DIR / f'{EXPERIMENT_NAME}_best.pt'

with open(log_path, 'w', newline='') as f:
    csv.writer(f).writerow(
        ['epoch', 'train_loss', 'train_acc', 'val_loss',
         'val_acc', 'val_macro_f1', 'val_balanced_acc']
    )

history = {k: [] for k in ['epoch', 'train_loss', 'train_acc',
                            'val_loss', 'val_acc', 'val_macro_f1', 'val_balanced_acc']}
best_val_f1 = -1.0
early_stopper = EarlyStopping(patience=PATIENCE)
t0 = time.time()

for epoch in range(1, EPOCHS + 1):
    ep_start = time.time()
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
    val_loss, val_acc, val_f1, val_bal, _, _ = evaluate(model, val_loader, criterion, DEVICE)
    scheduler.step(val_loss)

    history['epoch'].append(epoch)
    history['train_loss'].append(tr_loss)
    history['train_acc'].append(tr_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_macro_f1'].append(val_f1)
    history['val_balanced_acc'].append(val_bal)

    with open(log_path, 'a', newline='') as f:
        csv.writer(f).writerow(
            [epoch, tr_loss, tr_acc, val_loss, val_acc, val_f1, val_bal]
        )

    marker = ''
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                    'val_macro_f1': val_f1}, ckpt_path)
        marker = '  <-- new best, saved'

    dt = time.time() - ep_start
    print(f'epoch {epoch:>2}/{EPOCHS}  '
          f'train_loss={tr_loss:.4f} acc={tr_acc:.4f}  '
          f'val_loss={val_loss:.4f} acc={val_acc:.4f} f1={val_f1:.4f}  '
          f'({dt:.1f}s){marker}')

    early_stopper.check_early_stop(val_loss)
    if early_stopper.stop_training:
        print(f'early stopping at epoch {epoch}')
        break

print(f'\\ntraining done in {(time.time()-t0)/60:.1f} min, best val_macro_f1={best_val_f1:.4f}')

## 11. Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(history['epoch'], history['train_loss'], marker='o', label='train')
axes[0].plot(history['epoch'], history['val_loss'],   marker='s', label='val')
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('loss')
axes[0].set_title('Loss - augmented seed=123'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history['epoch'], history['val_macro_f1'], color='green', marker='o')
axes[1].set_xlabel('epoch'); axes[1].set_ylabel('val macro-F1')
axes[1].set_title('Val macro-F1 - augmented seed=123'); axes[1].grid(alpha=0.3)
axes[1].set_ylim(0, 1)

plt.tight_layout()
out = PLOTS_DIR / 'seed_check_curves.png'
plt.savefig(out, dpi=120, bbox_inches='tight')
print(f'saved {out}')
plt.show()

## 12. Test-set evaluation

In [ ]:
ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
ep = ckpt['epoch']
vf = ckpt['val_macro_f1']
print(f'loaded epoch {ep}  (val macro-F1 = {vf:.4f})')

test_loss, test_acc, test_f1, test_bal, test_preds, test_labels = evaluate(
    model, test_loader, criterion, DEVICE,
)
print(f'test_loss={test_loss:.4f}  test_acc={test_acc:.4f}  '
      f'test_macro_f1={test_f1:.4f}  test_balanced_acc={test_bal:.4f}')

print('\\nClassification report:')
print(classification_report(test_labels, test_preds,
                            labels=list(range(N_CLASSES)),
                            target_names=CLASS_NAMES,
                            zero_division=0, digits=3))

## 13. Seed-variance summary

The headline of this notebook. Compares the Phase 10 result (seed=42) against this run (seed=123) class by class. The "diff" column is what matters — if all the diffs are within roughly ±0.05 the original finding is solid; if any are bigger it's noisy.

In [ ]:
# Per-class recall from this seed=123 run
rec_seed123 = recall_score(test_labels, test_preds,
                           labels=list(range(N_CLASSES)),
                           average=None, zero_division=0)

# Pull Phase 10 (seed=42) results
phase10_csv = LOGS_DIR / 'comparison_baseline_vs_augmented.csv'
assert phase10_csv.exists(), f'expected Phase 10 comparison at {phase10_csv}'
p10 = pd.read_csv(phase10_csv)

rows = []
for i, name in enumerate(CLASS_NAMES):
    row_p10 = p10[p10['class'] == name].iloc[0]
    rows.append({
        'class': name,
        'phase10_seed42_recall':  row_p10['augmented_recall'],
        'seed123_recall':         rec_seed123[i],
        'seed_diff':              rec_seed123[i] - row_p10['augmented_recall'],
    })

# Overall row
row_overall = p10[p10['class'] == 'OVERALL'].iloc[0]
p10_overall = row_overall['augmented_recall']
rows.append({
    'class': 'OVERALL',
    'phase10_seed42_recall':  p10_overall,
    'seed123_recall':         test_f1,
    'seed_diff':              test_f1 - p10_overall,
})

summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False, float_format=lambda v: f'{v:.3f}'))

out_csv = LOGS_DIR / 'seed_variance_summary.csv'
summary_df.to_csv(out_csv, index=False)
print(f'\nsaved {out_csv}')

# Sentence to copy into the report
overall_diff = test_f1 - p10_overall
print('\n--- One-sentence report copy ---')
print(f'Re-running the Phase 10 augmented recipe with seed=123 reached test')
print(f'macro-F1 = {test_f1:.3f} (seed=42 reached {p10_overall:.3f},')
print(f'difference = {overall_diff:+.3f}), confirming the augmentation finding is')

if abs(overall_diff) < 0.03:
    msg = 'robust to seed.'
elif abs(overall_diff) < 0.05:
    msg = 'moderately seed-sensitive; we report this alongside the headline number.'
else:
    msg = 'sensitive to seed; small recipe differences should be interpreted cautiously.'
print(msg)

## 14. Wrap-up

This is the last training experiment for the project. From here it is report-writing only.

**Three places this result lands in the final report:**

1. In the **methodology / experimental setup** section: *"all training runs use a fixed seed (42), with a seed-variance check at seed=123 reported in Section X.Y."*
2. In the **results section**, as a sub-section or table footnote attached to the Phase 10 augmentation number.
3. In the **discussion / limitations** section: explicit note on the magnitude of seed variance observed, and how that bounds the confidence of small (~2-3 pp) differences between recipes.